# Unit 1: Regression

In this notebook we cover three types of regression:
1. **OLS (Ordinary Least Squares)** — predicting a continuous outcome
2. **Logistic Regression** — predicting a binary outcome
3. **Multinomial Logistic Regression** — predicting a categorical outcome with 3+ classes

For each, we will:
- Explore the data before modeling
- Fit the model and interpret the output
- Visualize coefficients with a forest plot

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.datasets import fetch_california_housing
import matplotlib.pyplot as plt
import seaborn as sns

## Helper function: parse_coefs

This function extracts a coefficient dataframe from a statsmodels summary table, which we can then use for forest plots.

It works two ways:
1. Pass a **statsmodels results object** (for OLS / Logit)
2. Pass a **raw block of rows** (list of lists) from a summary table — useful for MNLogit, where the table mixes class headers with coefficient rows and you need to slice out each block yourself.

**Hint for MNLogit**: look at `results.summary().tables[1].data` to see the raw table. Row 0 is the header for the first class, then `n_features` rows of coefficients, then another header row, then the second block of coefficients. Use list slicing to grab each block.

In [ ]:
def parse_coefs(results_or_block, table_index=1, rows=None, drop_const=True):
    """Extract a coefficient dataframe for forest plotting."""
    if isinstance(results_or_block, list):
        R = pd.DataFrame(results_or_block,
                         columns=["vari", "coef", "std_err", "t", "p_val", "low_b", "up_b"])
        R[["coef", "std_err", "t", "p_val", "low_b", "up_b"]] = \
            R[["coef", "std_err", "t", "p_val", "low_b", "up_b"]].astype(float)
    else:
        R = results_or_block.summary().tables[table_index].data
        R = pd.DataFrame(R)
        R.columns = R.iloc[0]
        R = R.iloc[1:]
        if rows is not None:
            R = R.iloc[rows]
        R[R.columns[1:]] = R[R.columns[1:]].astype(float)
        R.columns = ["vari", "coef", "std_err", "t", "p_val", "low_b", "up_b"]

    if drop_const:
        R = R[R.vari.str.strip() != "const"]
    R = R.sort_values("coef", ascending=True).reset_index(drop=True)
    p = 0.1
    R["alpha"] = (R.p_val < 0.05) * (1 - p) + p
    return R

---
# 1. OLS Regression

We will predict **median house value** in California districts using the 1990 Census data.

### 1.1 Load and explore the data

In [ ]:
california = fetch_california_housing()

df = pd.DataFrame(california.data, columns=california.feature_names)
df['MedHouseVal'] = california.target

print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.describe().round(2)

Let's look at how the features relate to each other and to the target. A **correlation heatmap** is a quick way to spot which predictors are most associated with the outcome and whether any predictors are highly correlated with each other (multicollinearity).

In [ ]:
cols = ["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup", "MedHouseVal"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[cols].corr().round(2), annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Correlation matrix")
plt.tight_layout()

In [ ]:
# Iterate on data exploration

### 1.2 Fit the OLS model

In [ ]:
# Define IVs and DV
X = df[["MedInc", "HouseAge", "AveRooms", "AveBedrms", "Population", "AveOccup"]]
y = df['MedHouseVal']

# Add a constant (intercept)
X = sm.add_constant(X)

# Fit the OLS regression model
ols_model = sm.OLS(y, X)
ols_results = ols_model.fit()

print(ols_results.summary())

### 1.3 Forest plot

In [ ]:
R = parse_coefs(ols_results)

plt.hlines(R.index, R.low_b, R.up_b, alpha=R.alpha)
plt.scatter(R.coef, R.index, alpha=R.alpha)
plt.axvline(0, linestyle="--", color="black")
plt.yticks(R.index, R.vari, fontsize=13)
plt.xlabel("Coefficient estimate")
plt.title("OLS Coefficients — California Housing")
plt.tight_layout()

In [ ]:
R = parse_coefs(ols_results, drop_const=False)

plt.hlines(R.index, R.low_b, R.up_b, alpha=R.alpha)
plt.scatter(R.coef, R.index, alpha=R.alpha)
plt.axvline(0, linestyle="--", color="black")
plt.yticks(R.index, R.vari, fontsize=13)
plt.xlabel("Coefficient estimate")
plt.title("OLS Coefficients (with constant) — California Housing")
plt.tight_layout()

### 1.4 Residual diagnostics

In [ ]:
residuals = ols_results.resid

plt.hist(residuals, bins=50, edgecolor='white')
plt.xlabel('Residual')
plt.title('Distribution of Residuals')
plt.tight_layout()

---
# 2. Logistic Regression

Your turn: Using the Titanic dataset, predict whether a passenger **survived** (binary outcome).

- **IVs**: `pclass`, `age`, `sibsp`, `fare`, `adult_male`
- **DV**: `survived`

### 2.1 Load and explore

In [ ]:
df = sns.load_dataset('titanic')

x_cols = ['pclass', 'age', 'sibsp', 'fare', 'adult_male']
y_col = 'survived'

df = df[x_cols + [y_col]].dropna()
print(f"Shape after dropping NAs: {df.shape}")
print(f"\nSurvival rate: {df.survived.mean():.1%}")
df.head()

### 2.2 Fit the logistic regression model and plot the forest plot

In [ ]:
# Your code here

---
# 3. Multinomial Logistic Regression

The UCI Wine dataset (ID 109) contains chemical properties of wines from three cultivars (classes 1, 2, 3). We'll predict the wine class from a subset of chemical features using `sm.MNLogit`.

Unlike binary logistic regression, multinomial logit estimates a separate set of coefficients **for each class** (relative to a reference class).

Use these features: `['Alcohol', 'Malicacid', 'Ash', 'Nonflavanoid_phenols', 'Color_intensity', 'Alcalinity_of_ash']`

Then, plot the two sets of coefficients (class 2 vs 1 and class 3 vs 1) on the same forest plot. (For an example, see [this figure](https://academic.oup.com/view-large/figure/447695951/pgae130f3.tif))

### 3.1 Load and explore

In [ ]:
wine = fetch_ucirepo(id=109)

X = wine.data.features
X = X[['Alcohol', 'Malicacid', 'Ash', 'Nonflavanoid_phenols', 'Color_intensity', 'Alcalinity_of_ash']].astype(float)
y = wine.data.targets

print(f"Shape: {X.shape}")
print(f"\nClass distribution:\n{y.value_counts().sort_index().to_string()}")

### 3.2 Fit the multinomial model and plot forest plots

In [ ]:
# Your code here